
# Experiment: Cross-Experiment Error Analysis Deep Dive

Objective:
- Go beyond the first-pass failure labels and ask what *kind* of wrong answers the models produce across experiments.
- Separate subtle, target-language near-misses from harder failures like source copying, script failure, decoding collapse, and English task escape.
- Quantify how those families move with difficulty (`depth`, grammar size, agreement condition, target orthography) and show concrete examples of each.

Success criteria:
- A compact taxonomy that explains most wrong outputs.
- At least one figure that shows cross-experiment error-family shifts.
- Conditioned slices for `agreement` and `orthography`, plus representative examples.


In [ ]:
from __future__ import annotations

import importlib
import json
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

try:
    from wordfreq import zipf_frequency
except ImportError:
    zipf_frequency = None

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.precision", 3)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".project-root").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root via .project-root")


PROJECT_ROOT = find_project_root(Path.cwd())
CACHE_DIR = PROJECT_ROOT / "notebooks" / "cache" / "error-analysis"
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

aes = importlib.import_module("aesthetics")
FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN = aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN

FIGSIZE_STACKED = (aes.COLM_PAPER_WIDTH_IN, 2.2 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
FIGSIZE_HEATMAP = (
    aes.COLM_PAPER_WIDTH_IN,
    1.5 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN,
)
FIGSIZE_TWO_PANEL = (aes.COLM_PAPER_WIDTH_IN, 1.4 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
FIGSIZE_THREE_PANEL = (
    aes.COLM_PAPER_WIDTH_IN,
    1.45 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN,
)
FIG_SIZE_SINGLE = (aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLEROW_IN)

WORD_ORDER_EXP = "wordorder_large_exp"
ORTHOGRAPHY_EXP = "orthography_large_exp"
WORD_ORDER_DATASETS = ("wordorder_large_exp",)
ORTHOGRAPHY_DATASETS = ("orthography_large_exp",)

if WORD_ORDER_EXP not in WORD_ORDER_DATASETS:
    raise ValueError(
        f"WORD_ORDER_EXP must be one of {WORD_ORDER_DATASETS}, got {WORD_ORDER_EXP!r}"
    )
if ORTHOGRAPHY_EXP not in ORTHOGRAPHY_DATASETS:
    raise ValueError(
        f"ORTHOGRAPHY_EXP must be one of {ORTHOGRAPHY_DATASETS}, "
        f"got {ORTHOGRAPHY_EXP!r}"
    )

EXPERIMENT_ORDER = ["size", "wordorder", "agreement", "orthography"]
MODEL_ORDER = [
    model
    for model in aes.MODEL_ORDER
    if model.startswith("gpt-5") or "gemma-3" in model
]
ERROR_ORDER = [
    "word_order_error",
    "omission",
    "extra_words",
    "recall_error",
    "source_vocab_error",
    "english_vocab",
    "hallucinated_vocab",
    "orthography_error",
    "mixed_other",
]
FAMILY_ORDER = ERROR_ORDER
FAMILY_PALETTE = {
    "word_order_error": "#5B8FF9",
    "omission": "#F4A261",
    "extra_words": "#C77DFF",
    "recall_error": "#2A9D8F",
    "source_vocab_error": "#52B788",
    "english_vocab": "#E9C46A",
    "hallucinated_vocab": "#E76F51",
    "orthography_error": "#7AA6E8",
    "mixed_other": "#ADB5BD",
}
FAMILY_LABELS = {
    "word_order_error": "word order errors",
    "omission": "omission",
    "extra_words": "extra words",
    "recall_error": "recall errors",
    "source_vocab_error": "source vocab",
    "english_vocab": "english vocab",
    "hallucinated_vocab": "hallucinated vocab",
    "orthography_error": "orthography",
    "mixed_other": "other",
}
FAMILY_PALETTE_LABELS = {
    FAMILY_LABELS[family]: color for family, color in FAMILY_PALETTE.items()
}
META_RE = re.compile(
    (
        r"(?:translate|translation|cannot|cant|can not|unable|sorry|input|"
        r"source|sentence|grammar|scfg|exact|determine|provided|given|lexical|"
        r"token|unknown|parse|information|reliably|hand)"
    ),
    re.IGNORECASE,
)
LATIN_WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
ENGLISH_ZIPF_MIN = 3.5


def tokenize(text: str | float | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return text.split()


def normalized_alpha_tokens(text: str | float | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return [token.lower() for token in LATIN_WORD_RE.findall(text)]


def normalize_vocab_tokens(vocab: set[str] | None) -> set[str]:
    normalized: set[str] = set()
    for token in vocab or set():
        normalized.update(normalized_alpha_tokens(str(token)))
    return normalized


def build_english_lookup(
    token_lists: pd.Series, min_zipf: float = ENGLISH_ZIPF_MIN
) -> dict[str, bool]:
    unique_tokens = sorted(
        {token for tokens in token_lists for token in tokens if len(token) >= 2}
    )
    if zipf_frequency is None:
        return {token: False for token in unique_tokens}
    return {token: zipf_frequency(token, "en") >= min_zipf for token in unique_tokens}


def english_word_tokens(
    pred_alpha_tokens: list[str],
    target_vocab_alpha: set[str],
    english_lookup: dict[str, bool],
) -> list[str]:
    return [
        token
        for token in dict.fromkeys(pred_alpha_tokens)
        if english_lookup.get(token, False) and token not in target_vocab_alpha
    ]


def counter_overlap(left: list[str], right: list[str]) -> int:
    return sum((Counter(left) & Counter(right)).values())


def build_vocab_map() -> dict[str, set[str]]:
    vocab_map: dict[str, set[str]] = {}
    for grammar_path in sorted((DATA_DIR / "grammars").glob("grammar_*.json")):
        payload = json.loads(grammar_path.read_text())
        vocab: set[str] = set()
        for key in ["target_vocab", "target_vocab_tokens"]:
            value = payload.get(key)
            if isinstance(value, list):
                vocab.update(map(str, value))
        vocab_map[payload["grammar_name"]] = vocab
    return vocab_map


def family_share(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    counts = df.groupby(group_cols, observed=False).size().rename("count").reset_index()
    totals = (
        df.groupby(group_cols[:-1], observed=False).size().rename("total").reset_index()
    )
    merged = counts.merge(totals, on=group_cols[:-1], how="left")
    merged["share"] = merged["count"] / merged["total"]
    return merged


def pct(value: float) -> str:
    return f"{100 * value:.1f}%"


def family_label(family: str) -> str:
    return FAMILY_LABELS.get(family, family.replace("_", " "))


def family_label_list(families: list[str]) -> list[str]:
    return [family_label(family) for family in families]


def parse_error_tags(value: str | float | None) -> list[str]:
    if not isinstance(value, str) or not value.strip():
        return []
    return [
        tag.strip()
        for tag in value.split("|")
        if tag.strip() and tag.strip() != "exact_match"
    ]


vocab_map = build_vocab_map()
target_vocab_alpha_map = {
    grammar_name: normalize_vocab_tokens(vocab)
    for grammar_name, vocab in vocab_map.items()
}
rows_df = pd.read_csv(CACHE_DIR / "rows.csv")
if "dataset" not in rows_df.columns:
    rows_df["dataset"] = (
        rows_df["exp"]
        .map(
            {
                "wordorder": "wordorder_exp",
                "size": "size_exp",
                "agreement": "agreement_exp",
                "orthography": "orthography_exp",
            }
        )
        .fillna("unknown")
    )

cached_dataset_counts = (
    rows_df.groupby(["dataset", "exp"], observed=False)
    .size()
    .rename("rows")
    .reset_index()
)
rows_df = rows_df.loc[
    (
        (rows_df["exp"].astype(str) != "wordorder")
        | rows_df["dataset"].eq(WORD_ORDER_EXP)
    )
    & (
        (rows_df["exp"].astype(str) != "orthography")
        | rows_df["dataset"].eq(ORTHOGRAPHY_EXP)
    )
].copy()

rows_df["model_name"] = rows_df["fuzzy_model"]
rows_df["exp"] = pd.Categorical(
    rows_df["exp"], categories=EXPERIMENT_ORDER, ordered=True
)
rows_df["fuzzy_model"] = pd.Categorical(
    rows_df["fuzzy_model"], categories=MODEL_ORDER, ordered=True
)
rows_df["pred_tokens"] = rows_df["model_answer"].fillna("").map(tokenize)
rows_df["ref_tokens"] = rows_df["output_sentence"].fillna("").map(tokenize)
rows_df["src_tokens"] = rows_df["input_sentence"].fillna("").map(tokenize)
rows_df["pred_alpha_tokens"] = (
    rows_df["model_answer"].fillna("").map(normalized_alpha_tokens)
)
rows_df["target_vocab"] = rows_df["grammar_name"].map(vocab_map)
rows_df["target_vocab_alpha"] = rows_df["grammar_name"].map(target_vocab_alpha_map)
english_lookup = build_english_lookup(rows_df["pred_alpha_tokens"])
rows_df["english_word_tokens"] = rows_df.apply(
    lambda row: english_word_tokens(
        row["pred_alpha_tokens"],
        row["target_vocab_alpha"]
        if isinstance(row["target_vocab_alpha"], set)
        else set(),
        english_lookup,
    ),
    axis=1,
)
rows_df["contains_english_words"] = rows_df["english_word_tokens"].map(bool)
rows_df["english_word_count"] = rows_df["english_word_tokens"].map(len)
rows_df["target_recall"] = rows_df.apply(
    lambda row: 0.0
    if not row["ref_tokens"]
    else counter_overlap(row["pred_tokens"], row["ref_tokens"])
    / len(row["ref_tokens"]),
    axis=1,
)
rows_df["target_precision"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else counter_overlap(row["pred_tokens"], row["ref_tokens"])
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["source_share"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else counter_overlap(row["pred_tokens"], row["src_tokens"])
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["licensed_token_share"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else sum(
        token
        in (row["target_vocab"] if isinstance(row["target_vocab"], set) else set())
        for token in row["pred_tokens"]
    )
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["fully_licensed"] = rows_df["licensed_token_share"].eq(1.0)
rows_df["same_len"] = rows_df["pred_len"] == rows_df["ref_len"]
rows_df["meta_response"] = rows_df["model_answer"].fillna("").str.contains(META_RE)
rows_df["error_tags"] = rows_df["failure_tags_str"].map(parse_error_tags)
rows_df["primary_error"] = rows_df["error_tags"].map(
    lambda tags: tags[0] if tags else "exact_match"
)
wrong_df = rows_df.loc[~rows_df["exact_match"]].copy()
wrong_tag_df = wrong_df.drop(columns=["error_family"], errors="ignore").explode(
    "error_tags"
)
wrong_tag_df = wrong_tag_df.rename(columns={"error_tags": "error_family"})
wrong_tag_df = wrong_tag_df.loc[
    wrong_tag_df["error_family"].notna() & wrong_tag_df["error_family"].ne("")
].copy()
size_order = [
    f"{int(value):,}"
    for value in sorted(
        rows_df.loc[rows_df["exp"] == "size", "n_words"].dropna().unique()
    )
]
rows_df["size_bin"] = pd.Categorical(
    rows_df["n_words"].map(
        lambda value: f"{int(value):,}" if pd.notna(value) else np.nan
    ),
    categories=size_order,
    ordered=True,
)
wrong_df["size_bin"] = rows_df.loc[wrong_df.index, "size_bin"]
wrong_tag_df["size_bin"] = rows_df.loc[wrong_tag_df.index, "size_bin"].to_numpy()

print(f"Using WORD_ORDER_EXP={WORD_ORDER_EXP}")
print(f"Using ORTHOGRAPHY_EXP={ORTHOGRAPHY_EXP}")
print(
    f"Loaded {len(rows_df):,} rows and {len(wrong_df):,} wrong answers "
    f"after dataset filtering."
)
print(
    "Error categories:",
    ", ".join(
        family for family in FAMILY_ORDER if family in set(wrong_tag_df["error_family"])
    ),
)
print(
    "Wrong answers with English lexical intrusion:",
    pct(wrong_df["contains_english_words"].mean()),
)
selected_counts = cached_dataset_counts.loc[
    cached_dataset_counts["dataset"].isin([WORD_ORDER_EXP, ORTHOGRAPHY_EXP])
].copy()
print(selected_counts.sort_values(["exp", "dataset"]).to_string(index=False))

## Plan

Goals:
- Collapse the taxonomy to the current error categories used by `error_analysis.py`.
- Treat wrong answers as multi-tag rows rather than forcing each mistake into one exclusive bucket.
- Show the main cross-experiment comparison for `gpt-5` only, with one panel per experiment.

Metrics used here:
- `target_recall` / `target_precision`: multiset token overlap with the reference.
- `source_share`: how much of the prediction is copied from the source sentence.
- `licensed_token_share`: how much of the prediction stays inside the grammar's target-side vocabulary.
- `meta_response`: whether the model answers in English about the task instead of translating.

Dataset selectors:
- `WORD_ORDER_EXP` and `ORTHOGRAPHY_EXP` at the top of the setup cell choose which cached dataset backs the `wordorder` and `orthography` analyses.



## A Compact Error Taxonomy

The first-pass labels are useful, but they over-separate some patterns and under-separate others. The family view below answers a more causal question: when the model is wrong, is it still behaving like a translator, or has it switched into some other failure mode?

Alongside the mutually exclusive taxonomy, I also track one overlapping diagnostic: **English lexical intrusion**. This fires when the model's final answer contains common English words that are *not* licensed by that grammar's target vocabulary, so accidental homographs in the target lexicon do not count.

In [ ]:
family_share_df = family_share(wrong_tag_df, ["exp", "fuzzy_model", "error_family"])
dominant_family_df = (
    family_share_df.sort_values(
        ["exp", "fuzzy_model", "share"], ascending=[True, True, False]
    )
    .groupby(["exp", "fuzzy_model"], as_index=False, observed=False)
    .head(1)
    .rename(
        columns={
            "error_family": "dominant_wrong_family",
            "share": "dominant_wrong_share",
        }
    )
)

english_intrusion_df = (
    wrong_df.groupby(["exp", "fuzzy_model"], observed=False)["contains_english_words"]
    .mean()
    .reset_index(name="english_words_within_wrong")
)

topline_df = (
    rows_df.groupby(["exp", "fuzzy_model"], observed=False)
    .agg(
        rows=("custom_id", "size"),
        exact_match=("exact_match", "mean"),
        bow_match=("bow_match", "mean"),
        mean_ref_len=("ref_len", "mean"),
    )
    .reset_index()
    .merge(dominant_family_df, on=["exp", "fuzzy_model"], how="left")
    .merge(english_intrusion_df, on=["exp", "fuzzy_model"], how="left")
    .fillna({"english_words_within_wrong": 0.0})
)

display(
    topline_df.assign(
        exact_match=lambda df: (100 * df["exact_match"]).round(1),
        bow_match=lambda df: (100 * df["bow_match"]).round(1),
        dominant_wrong_family=lambda df: df["dominant_wrong_family"].map(family_label),
        dominant_wrong_share=lambda df: (100 * df["dominant_wrong_share"]).round(1),
        english_words_within_wrong=lambda df: (
            100 * df["english_words_within_wrong"]
        ).round(1),
        mean_ref_len=lambda df: df["mean_ref_len"].round(1),
    )[
        [
            "exp",
            "fuzzy_model",
            "rows",
            "exact_match",
            "bow_match",
            "dominant_wrong_family",
            "dominant_wrong_share",
            "english_words_within_wrong",
            "mean_ref_len",
        ]
    ].sort_values(["exp", "fuzzy_model"])
)

In [ ]:
taxonomy_descriptions = {
    "word_order_error": (
        "The prediction contains the right target words but places them "
        "in the wrong order."
    ),
    "omission": (
        "The answer drops a required token or span from the reference " "translation."
    ),
    "extra_words": (
        "The answer adds target-side material beyond the reference " "translation."
    ),
    "recall_error": (
        "The model uses target-language vocabulary, but retrieves the "
        "wrong lexical item or inflected form."
    ),
    "source_vocab_error": (
        "Source-language words leak into the translation instead of being "
        "translated."
    ),
    "english_vocab": (
        "English words appear in the translation even though they are not "
        "licensed source or target vocabulary."
    ),
    "hallucinated_vocab": (
        "The answer invents vocabulary that is not in the target lexicon, "
        "the source lexicon, or plausible English intrusion."
    ),
    "orthography_error": (
        "The answer uses the wrong script or violates the target " "orthography policy."
    ),
    "mixed_other": (
        "Residual wrong answers that do not fit the cleaner categories " "above."
    ),
}

taxonomy_order = [
    family for family in FAMILY_ORDER if family in set(wrong_tag_df["error_family"])
]
taxonomy_rows = []
for family in taxonomy_order:
    example = (
        wrong_tag_df.loc[wrong_tag_df["error_family"] == family]
        .sort_values(
            [
                "target_recall",
                "licensed_token_share",
                "source_share",
                "exp",
                "model_name",
                "depth",
                "n_words",
            ],
            ascending=[False, False, True, True, True, True, True],
            na_position="last",
        )
        .head(1)
    )
    if example.empty:
        continue
    row = example.iloc[0]
    taxonomy_rows.append(
        {
            "error_family": family,
            "description": taxonomy_descriptions[family],
            "exp": row["exp"],
            "model": row["model_name"],
            "failure_mode": row["failure_mode"],
            "input_sentence": row["input_sentence"],
            "output_sentence": row["output_sentence"],
            "model_answer": row["model_answer"],
        }
    )

glossary_sections = [
    "### Taxonomy Glossary: current error categories plus one example per category"
]
for item in taxonomy_rows:
    glossary_sections.extend(
        [
            f"#### `{family_label(item['error_family'])}`",
            item["description"],
            "",
            f"Example: `{item['exp']}` | `{item['model']}` | `{item['failure_mode']}`",
            f"- Input: `{item['input_sentence']}`",
            f"- Target: `{item['output_sentence']}`",
            f"- Prediction: `{item['model_answer']}`",
            "",
        ]
    )

display(Markdown("\n".join(glossary_sections)))

In [ ]:
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"

In [ ]:
plot_df = family_share(
    wrong_tag_df.loc[wrong_tag_df["fuzzy_model"].astype(str) == "gpt-5"],
    ["exp", "error_family"],
)
present_families = [
    family for family in FAMILY_ORDER if family in set(plot_df["error_family"])
]
plot_df["error_family"] = pd.Categorical(
    plot_df["error_family"], categories=present_families, ordered=True
)

taxonomy_experiment_order = ["size", "wordorder", "agreement", "orthography"]
experiment_panels = [
    exp
    for exp in taxonomy_experiment_order
    if (plot_df["exp"].astype(str) == exp).any()
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN),
    sharex=True,
)
axes = np.atleast_1d(axes).reshape(2, 2)

y_positions = np.arange(len(present_families))
bar_height = 0.72

for panel_idx, exp in enumerate(experiment_panels):
    row_idx, col_idx = divmod(panel_idx, 2)
    ax = axes[row_idx, col_idx]
    exp_plot_df = plot_df.loc[plot_df["exp"].astype(str) == exp].copy()
    exp_plot_df = (
        exp_plot_df.set_index("error_family")["share"]
        .reindex(present_families, fill_value=0.0)
        .rename("share")
        .reset_index()
    )
    exp_plot_df["error_family_label"] = exp_plot_df["error_family"].map(family_label)

    ax.barh(
        y_positions,
        exp_plot_df["share"],
        height=bar_height,
        color=[FAMILY_PALETTE[family] for family in exp_plot_df["error_family"]],
        edgecolor="white",
        linewidth=0.8,
    )

    exp_label = {
        "wordorder": "Word Order",
        "size": "Size",
        "agreement": "Agreement",
        "orthography": "Orthography",
    }.get(exp, exp.title())
    ax.set_title(exp_label, loc="left", fontweight="normal")
    ax.set_xlim(0, 1.0)
    ax.set_ylim(-0.5, len(present_families) - 0.5)
    ax.set_yticks(y_positions)
    if col_idx == 0:
        ax.set_yticklabels(exp_plot_df["error_family_label"])
    else:
        ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_ylabel("")
    ax.tick_params(axis="y", pad=4)
    if row_idx == 1:
        ax.set_xlabel("Share of wrong answers")
    else:
        ax.set_xlabel("")
    sns.despine(ax=ax, left=False, bottom=False)

for panel_idx in range(len(experiment_panels), 4):
    row_idx, col_idx = divmod(panel_idx, 2)
    axes[row_idx, col_idx].set_visible(False)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0.1, hspace=0.3)
plt.show()

aes.save_figure(FIGURES_DIR / "gpt-5_error_taxonomy_by_experiment", fig=fig)

Two immediate takeaways:
- With the multi-tag view, a single wrong answer can contribute to several categories at once. The bars therefore show the share of wrong answers carrying each tag, not an exclusive partition of the error mass.
- For `gpt-5`, the cross-experiment pattern is still legible: `agreement` stays dominated by target-side recall problems, while `orthography` surfaces script and form issues much more directly.



In [ ]:
portrait_df = (
    wrong_tag_df.groupby("error_family", observed=False)
    .agg(
        target_recall=("target_recall", "mean"),
        target_precision=("target_precision", "mean"),
        source_share=("source_share", "mean"),
        licensed_token_share=("licensed_token_share", "mean"),
        meta_response=("meta_response", "mean"),
        contains_english_words=("contains_english_words", "mean"),
    )
    .reindex(
        [
            family
            for family in FAMILY_ORDER
            if family in set(wrong_tag_df["error_family"])
        ]
    )
)

metric_labels = {
    "target_recall": "target recall",
    "target_precision": "target precision",
    "source_share": "source share",
    "licensed_token_share": "licensed share",
    "meta_response": "meta response",
    "contains_english_words": "english words",
}
portrait_display_df = (100 * portrait_df).round(1).rename(columns=metric_labels)
portrait_display_df.index = portrait_display_df.index.map(family_label)

display(portrait_display_df)

fig, ax = plt.subplots(figsize=FIGSIZE_HEATMAP)
portrait_heatmap_df = portrait_df.rename(index=FAMILY_LABELS, columns=metric_labels)
sns.heatmap(
    portrait_heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Mean value"},
    ax=ax,
)
ax.set_title(
    "Error-category portraits: overlap, source copying, licensing, and English leakage"
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Difficulty Gradients

The next question is not just *what* the error families are, but *when* they appear. The plots below separate two different failure trajectories:
- a graceful degradation trajectory, where predictions remain target-like but drift lexically;
- a brittle trajectory, where the model leaks source words or abandons the task in English.


In [ ]:
wordorder_depth_df = family_share(
    wrong_tag_df.loc[wrong_tag_df["exp"].astype(str) == "wordorder"],
    ["fuzzy_model", "depth", "error_family"],
)
plot_families = [
    "word_order_error",
    "recall_error",
    "source_vocab_error",
    "hallucinated_vocab",
    "orthography_error",
]
wordorder_plot_df = wordorder_depth_df.loc[
    wordorder_depth_df["error_family"].isin(plot_families)
].copy()
wordorder_plot_df["share_pct"] = 100 * wordorder_plot_df["share"]
wordorder_plot_df["error_family_label"] = wordorder_plot_df["error_family"].map(
    family_label
)
plot_palette = {
    family_label(family): FAMILY_PALETTE[family] for family in plot_families
}
plot_order = family_label_list(plot_families)

plot_models = [
    model
    for model in MODEL_ORDER
    if model in set(wordorder_plot_df["fuzzy_model"].astype(str))
]

fig, axes = plt.subplots(
    1,
    len(plot_models),
    figsize=(
        max(1.0, len(plot_models) / 2) * aes.COLM_PAPER_WIDTH_IN,
        2 * aes.FIG_HEIGHT_SINGLE_ROW_IN,
    ),
    sharey=True,
)
axes = np.atleast_1d(axes)
for ax, model in zip(axes, plot_models):
    subset = wordorder_plot_df.loc[
        wordorder_plot_df["fuzzy_model"].astype(str) == model
    ]
    sns.lineplot(
        data=subset,
        x="depth",
        y="share_pct",
        hue="error_family_label",
        hue_order=plot_order,
        palette=plot_palette,
        marker="o",
        linewidth=2,
        ax=ax,
    )
    ax.set_title(aes.MODEL_DISPLAY_NAMES.get(model, model))
    ax.set_xlabel("Sentence depth")
    ax.set_ylabel("Share of wrong answers (%)")
    ax.set_ylim(0, 100)
    if ax is axes[-1]:
        ax.legend(title="", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.get_legend().remove()

plt.tight_layout()
plt.show()

In [ ]:
size_family_df = family_share(
    wrong_tag_df.loc[wrong_tag_df["exp"].astype(str) == "size"],
    ["fuzzy_model", "size_bin", "error_family"],
)
plot_families = [
    "recall_error",
    "omission",
    "source_vocab_error",
    "hallucinated_vocab",
    "orthography_error",
]
size_plot_df = size_family_df.loc[
    size_family_df["error_family"].isin(plot_families)
].copy()
size_plot_df["share_pct"] = 100 * size_plot_df["share"]
size_plot_df["error_family_label"] = size_plot_df["error_family"].map(family_label)
plot_palette = {
    family_label(family): FAMILY_PALETTE[family] for family in plot_families
}
plot_order = family_label_list(plot_families)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_THREE_PANEL, sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    subset = size_plot_df.loc[size_plot_df["fuzzy_model"].astype(str) == model]
    if subset.empty:
        ax.set_visible(False)
        continue
    sns.lineplot(
        data=subset,
        x="size_bin",
        y="share_pct",
        hue="error_family_label",
        hue_order=plot_order,
        palette=plot_palette,
        marker="o",
        linewidth=2,
        ax=ax,
    )
    ax.set_title(model)
    ax.set_xlabel("Target lexicon size (`n_words`)")
    ax.set_ylabel("Share of wrong answers (%)")
    ax.tick_params(axis="x", rotation=45)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.get_legend().remove()

fig.suptitle(
    (
        "Grammar size changes which error categories co-occur, "
        "especially for smaller models"
    ),
    y=1.08,
)
plt.tight_layout()
plt.show()


## Agreement: Well-Formed But Wrong

`agreement` is the clearest case where the family view changes the interpretation. The main question here is whether failures are random, or whether the model is producing a target-language sentence with the wrong feature binding.


In [ ]:
agreement_df = rows_df.loc[rows_df["exp"].astype(str) == "agreement"].copy()
agreement_tag_df = wrong_tag_df.loc[
    wrong_tag_df["exp"].astype(str) == "agreement"
].copy()
agreement_summary_df = (
    agreement_df.groupby(["fuzzy_model", "agreement_condition"], observed=False)
    .apply(
        lambda group: pd.Series(
            {
                "rows": len(group),
                "exact_match": group["exact_match"].mean(),
                "fully_licensed_within_wrong": group.loc[
                    ~group["exact_match"], "fully_licensed"
                ].mean(),
                "licensed_same_len_within_wrong": (
                    group.loc[~group["exact_match"], "fully_licensed"]
                    & group.loc[~group["exact_match"], "same_len"]
                ).mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
recall_df = family_share(
    agreement_tag_df, ["fuzzy_model", "agreement_condition", "error_family"]
)
recall_df = recall_df.loc[
    recall_df["error_family"] == "recall_error",
    [
        "fuzzy_model",
        "agreement_condition",
        "share",
    ],
].rename(columns={"share": "recall_error_within_wrong"})
agreement_summary_df = agreement_summary_df.merge(
    recall_df, on=["fuzzy_model", "agreement_condition"], how="left"
).fillna({"recall_error_within_wrong": 0.0})

agreement_display_df = agreement_summary_df.copy()
for col in [
    "exact_match",
    "fully_licensed_within_wrong",
    "licensed_same_len_within_wrong",
    "recall_error_within_wrong",
]:
    agreement_display_df[col] = (100 * agreement_display_df[col]).round(1)
agreement_display_df = agreement_display_df.rename(
    columns={
        "exact_match": "exact match",
        "fully_licensed_within_wrong": "fully licensed within wrong",
        "licensed_same_len_within_wrong": "same-length licensed within wrong",
        "recall_error_within_wrong": "recall error within wrong",
    }
)

display(agreement_display_df.sort_values(["fuzzy_model", "agreement_condition"]))

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_THREE_PANEL, sharey=False)
for ax, metric, title in zip(
    axes,
    ["exact_match", "fully_licensed_within_wrong", "recall_error_within_wrong"],
    [
        "Exact-match rate",
        "Wrong outputs that stay fully inside target vocab",
        "Wrong outputs tagged as recall errors",
    ],
):
    sns.barplot(
        data=agreement_summary_df,
        x="agreement_condition",
        y=metric,
        hue="fuzzy_model",
        palette=sns.color_palette("Reds", n_colors=3),
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Rate")
    ax.tick_params(axis="x", rotation=30)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, loc="upper right")
    else:
        ax.get_legend().remove()

fig.suptitle(
    (
        "Agreement errors are usually target-side retrieval mistakes, "
        "not source copying"
    ),
    y=1.08,
)
plt.tight_layout()
plt.show()


Interpretation:
- Translating *into* agreement (`NoAgr -> Agr`) is the hardest setting in exact-match terms.
- But even there, many wrong answers are still fully licensed target strings, which suggests a feature-binding problem rather than a total breakdown.
- `NoAgr -> NoAgr` is the easiest condition and also the one where wrong answers are most often fully licensed and same-length.



## Orthography: Script Handling Versus Translation

`orthography` is useful because it cleanly separates lexical/structural translation from writing-system realization. The table and plots below show that the two models fail in qualitatively different ways.


In [ ]:
orth_df = rows_df.loc[rows_df["exp"].astype(str) == "orthography"].copy()
orth_tag_df = wrong_tag_df.loc[wrong_tag_df["exp"].astype(str) == "orthography"].copy()
orth_summary_df = (
    orth_df.groupby(["fuzzy_model", "target_orthography"], observed=False)
    .apply(
        lambda group: pd.Series(
            {
                "rows": len(group),
                "exact_match": group["exact_match"].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
for tag in ["orthography_error", "source_vocab_error", "english_vocab"]:
    tag_share_df = family_share(
        orth_tag_df, ["fuzzy_model", "target_orthography", "error_family"]
    )
    tag_share_df = tag_share_df.loc[
        tag_share_df["error_family"] == tag,
        [
            "fuzzy_model",
            "target_orthography",
            "share",
        ],
    ].rename(columns={"share": tag})
    orth_summary_df = orth_summary_df.merge(
        tag_share_df, on=["fuzzy_model", "target_orthography"], how="left"
    )
orth_summary_df = orth_summary_df.fillna(0.0)

orth_summary_plot_df = orth_summary_df.loc[
    orth_summary_df["fuzzy_model"].astype(str).isin(MODEL_ORDER)
].copy()
orth_summary_plot_df["fuzzy_model"] = orth_summary_plot_df["fuzzy_model"].astype(str)
orth_display_df = orth_summary_plot_df.copy()
for col in ["exact_match", "orthography_error", "source_vocab_error", "english_vocab"]:
    orth_display_df[col] = (100 * orth_display_df[col]).round(1)
orth_display_df = orth_display_df.rename(
    columns={
        "exact_match": "exact match",
        "orthography_error": "orthography",
        "source_vocab_error": "source vocab",
        "english_vocab": "english vocab",
    }
)

display(orth_display_df.sort_values(["fuzzy_model", "target_orthography"]))

fig, axes = plt.subplots(
    1,
    3,
    figsize=(1.6 * aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN),
    sharey=False,
)
for ax, metric, title in zip(
    axes,
    ["exact_match", "orthography_error", "source_vocab_error"],
    ["Exact-match rate", "Orthography-tag rate", "Source-vocab rate"],
):
    sns.barplot(
        data=orth_summary_plot_df,
        x="target_orthography",
        y=metric,
        hue="fuzzy_model",
        hue_order=[
            model
            for model in MODEL_ORDER
            if model in set(orth_summary_plot_df["fuzzy_model"])
        ],
        palette=aes.PALETTE_MODELS,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Rate")
    ax.tick_params(axis="x", rotation=30)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, loc="upper right")
    else:
        ax.get_legend().remove()

fig.suptitle(
    "Orthography separates script/form mistakes from broader lexical failure",
    y=1.08,
)
plt.tight_layout()
plt.show()

## Literature Grounding

The taxonomy above is deliberately lightweight, but it tracks familiar distinctions from prior MT and generation work:

- **MQM** frames translation errors as multidimensional quality problems rather than a single scalar score, which is useful here because exact-match collapses near-miss and multi-tag failures together. See the MQM overview and shared-task framing in [Lommel et al. (2014)](https://aclanthology.org/W14-3302/).
- **Adequacy versus target-side fluency**: [Tu et al. (2017), *Context Gates for Neural Machine Translation*](https://aclanthology.org/Q17-1007/) is a useful reminder that outputs can remain fluent while becoming weakly conditioned on the source. That is close to the `recall error` story here.
- **Hallucination / detachment from the source**: [Guerreiro et al. (2023), *Hallucinations in Neural Machine Translation*](https://aclanthology.org/2023.findings-acl.283/) is relevant for the `hallucinated vocab`, `source vocab`, and `english vocab` categories.
- **Orthographic control**: script, diacritic, and transliteration failures fit naturally with the `orthography error` category, which isolates surface-form violations from lexical recall mistakes.

The distinctive result in this repository is that many failures are **not** classical hallucinations. Especially in `agreement`, the models often produce a target-language sentence that is almost right, but not semantically or morphologically correct.



In [ ]:
EXAMPLE_COLUMNS = [
    "exp",
    "model_name",
    "failure_mode",
    "error_family",
    "depth",
    "n_words",
    "target_recall",
    "licensed_token_share",
    "source_share",
    "input_sentence",
    "output_sentence",
    "model_answer",
]
ENGLISH_EXAMPLE_COLUMNS = EXAMPLE_COLUMNS + ["english_word_tokens"]


def show_examples(family: str, n: int = 3, exp: str | None = None) -> pd.DataFrame:
    subset = wrong_tag_df.loc[wrong_tag_df["error_family"] == family].copy()
    if exp is not None:
        subset = subset.loc[subset["exp"].astype(str) == exp]
    subset = subset.sort_values(
        ["model_name", "exp", "depth", "n_words"], na_position="last"
    )
    subset = subset.head(n)
    display_df = subset[EXAMPLE_COLUMNS].copy()
    display_df["error_family"] = display_df["error_family"].map(family_label)
    for col in ["target_recall", "licensed_token_share", "source_share"]:
        display_df[col] = display_df[col].round(2)
    return display_df


def show_english_examples(
    n_per_family: int = 1, exp: str | None = None
) -> pd.DataFrame:
    subset = wrong_tag_df.loc[wrong_tag_df["contains_english_words"]].copy()
    if exp is not None:
        subset = subset.loc[subset["exp"].astype(str) == exp]
    subset = subset.sort_values(
        [
            "english_word_count",
            "target_recall",
            "model_name",
            "exp",
            "depth",
            "n_words",
        ],
        ascending=[False, False, True, True, True, True],
        na_position="last",
    )
    subset = (
        subset.groupby("error_family", observed=False, as_index=False)
        .head(n_per_family)
        .sort_values(["error_family", "model_name", "exp"], na_position="last")
    )
    display_df = subset[ENGLISH_EXAMPLE_COLUMNS].copy()
    display_df["error_family"] = display_df["error_family"].map(family_label)
    for col in ["target_recall", "licensed_token_share", "source_share"]:
        display_df[col] = display_df[col].round(2)
    return display_df


example_specs = [
    ("recall_error", "agreement", "Recall error examples (agreement)"),
    ("word_order_error", "wordorder", "Word-order examples"),
    ("source_vocab_error", None, "Source-vocab examples"),
    ("orthography_error", "orthography", "Orthography examples"),
    ("omission", None, "Omission examples"),
]

for family, exp, title in example_specs:
    display(Markdown(f"### {title}"))
    display(show_examples(family=family, exp=exp))

display(Markdown("### English-word intrusion examples across categories"))
display(show_english_examples())

## Takeaways

- The biggest correction to the first-pass story is that many errors are **target-language recall errors**, not generic hallucinations. This is strongest in `agreement`, where wrong outputs often stay fully inside the target lexicon.
- Difficulty changes the *kind* of error. Stronger models tend to degrade into recall errors or word-order mistakes, while weaker settings pick up more source-vocab, English-vocab, and hallucinated-vocab tags.
- Because the categories are now multi-tag, the same row can simultaneously be an `orthography error` and a `hallucinated vocab` case. The deep-dive plots should therefore be read as overlapping shares, not partitions.
- `orthography` is still genuinely special: it reveals a separate axis of failure around script and diacritic realization, which would be easy to misread as ordinary lexical error if we only looked at exact match.
- For follow-up work, the most promising next step is to split the large `recall error` bucket into finer lexical, feature-binding, and scope/attachment subtypes once we need that resolution again.

